# 03 · RAG: índice, busca e resposta

Usa os trechos gerados no notebook `02_ingestao`:

1. gera vetores **densos e esparsos** com o **BGE-M3** e monta o índice no **Qdrant** local;
2. faz a busca **híbrida** (os dois tipos de vetor fundidos por RRF);
3. mede o **recall@5** com as perguntas de `data/avaliacao/perguntas_desempate.csv` e sugere o limiar de recusa;
4. gera respostas com o modelo-base, citando a fonte, nos modos público e pesquisador.

Precisa de GPU T4 só na seção 6, a geração. As seções 1 a 5 rodam também em CPU, mais devagar.

## 1. Instalação

In [ ]:
%%capture
!pip install --upgrade unsloth unsloth_zoo qdrant-client

## 2. Preparação: Drive, repositório e módulo

In [ ]:
import os, subprocess, sys

# Pasta dos dados no seu Google Drive (a que tem o documentos.csv).
PASTA_DADOS = "/content/drive/MyDrive/refauna-slm/dados"
REPO_URL = "https://github.com/iamrodrigocoelho/refauna-slm"

from google.colab import drive
drive.mount("/content/drive")

if os.path.exists("refauna-slm"):
    subprocess.run(["git", "-C", "refauna-slm", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL], check=True)
sys.path.insert(0, "refauna-slm/src")

import importlib
import refauna_rag as rr
importlib.reload(rr)
print("Dados em:", PASTA_DADOS)

In [ ]:
trechos = rr.carregar_trechos(f"{PASTA_DADOS}/processado/trechos.jsonl")
documentos = sorted({t["arquivo"] for t in trechos})
print(f"{len(trechos)} trechos de {len(documentos)} documentos")

## 3. Índice

In [ ]:
import time

embedder = rr.EmbedderBGEM3()        # baixa o BGE-M3 na primeira vez (cerca de 2 GB)
inicio = time.time()
cliente = rr.criar_indice(trechos, embedder, caminho="/content/qdrant")
print(f"Índice criado em {time.time() - inicio:.0f} s")

## 4. Busca de exemplo

Troque a pergunta e veja o que volta. `acessos=("publico",)` simula a demo pública, que nunca mostra documentos internos.

In [ ]:
PERGUNTA = "Quantos filhotes de cutia foram observados depois da reintrodução?"

resultado, similaridade = rr.buscar(cliente, embedder, PERGUNTA, k=5)
print(f"Maior similaridade densa: {similaridade:.3f}\n")
for i, t in enumerate(resultado, 1):
    print(f"[{i}] {rr.citacao_curta(t)} · {t['secao'][:60]} · RRF {t['pontuacao']:.3f}")
    print("    ", t["texto"][:250].replace("\n", " "), "\n")

## 5. Avaliar a busca

Para cada pergunta cuja fonte já está no índice, confere se o trecho esperado aparece entre os 5 primeiros. Perguntas de documentos ainda não baixados ficam de fora da conta.

A coluna `similaridade` mostra o quanto a melhor resposta se parece com a pergunta. As perguntas fora do escopo devem ficar abaixo das demais: é daí que sai o limiar de recusa.

In [ ]:
import pandas as pd

perguntas = pd.read_csv("refauna-slm/data/avaliacao/perguntas_desempate.csv")
fontes_indexadas = {t["fonte"] for t in trechos if t["fonte"]}
avaliacao = rr.avaliar_busca(perguntas, cliente, embedder, fontes_indexadas, k=5)

avaliaveis = avaliacao[avaliacao["avaliavel"].astype(bool)].astype({"encontrado": bool})
if len(avaliaveis):
    print(f"Recall@5: {avaliaveis['encontrado'].mean():.0%} ({int(avaliaveis['encontrado'].sum())} de {len(avaliaveis)} perguntas)")
else:
    print("Nenhuma pergunta tem fonte no índice ainda. Baixe os documentos citados nas perguntas.")
avaliacao

In [ ]:
# Sugestão de limiar: entre a maior similaridade das perguntas fora do escopo
# e as similaridades mais baixas das perguntas que têm resposta no índice.
fora = avaliacao.loc[avaliacao["tipo"] == "fora_escopo", "similaridade"]
dentro = avaliaveis["similaridade"] if len(avaliaveis) else pd.Series(dtype=float)
print("Fora do escopo:", fora.round(3).tolist())
print("Com resposta no índice:", dentro.round(3).tolist())
if len(fora) and len(dentro):
    LIMIAR = round((fora.max() + dentro.quantile(0.1)) / 2, 3)
else:
    LIMIAR = 0.45
print("Limiar de recusa:", LIMIAR)

Se as faixas se sobrepõem, o limiar sozinho não separa bem as perguntas, e a recusa vai depender do modelo. Isso é parte do que o fine-tuning deve ensinar. Registre o recall e o limiar em `docs/diario.md`.

## 6. Gerar respostas

Escolha o modelo-base vencedor do notebook `01`. O modelo recebe os 5 trechos numerados e deve citar `[n]`.

In [ ]:
import re, torch
from unsloth import FastModel

MODELOS = {
    "gemma4_e4b": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    "qwen35_4b": "Qwen/Qwen3.5-4B",
}
MODELO = "gemma4_e4b"

model, tokenizer = FastModel.from_pretrained(model_name=MODELOS[MODELO], max_seq_length=8192, load_in_4bit=True)
FastModel.for_inference(model)

def gerar(mensagens, max_new_tokens=500):
    kwargs = dict(add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    try:
        entrada = tokenizer.apply_chat_template(mensagens, enable_thinking=False, **kwargs)
    except TypeError:
        entrada = tokenizer.apply_chat_template(mensagens, **kwargs)
    entrada = entrada.to("cuda")
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    texto = tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.S).strip()

print(f"Memória de vídeo em uso: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
resposta = rr.responder(PERGUNTA, cliente, embedder, gerar, modo="publico", limiar=LIMIAR)
print(resposta["resposta"], "\n")
print("Fontes:")
print("\n".join(resposta["fontes"]) or "(nenhuma citada)")

## 7. Rodar todas as perguntas nos dois modos e salvar

In [ ]:
import os

linhas = []
for p in perguntas.to_dict("records"):
    for modo in ["publico", "pesquisador"]:
        r = rr.responder(p["pergunta"], cliente, embedder, gerar, modo=modo, limiar=LIMIAR)
        linhas.append({
            "id": p["id"], "tipo": p["tipo"], "modo": modo, "pergunta": p["pergunta"],
            "resposta_esperada": p["resposta_esperada"], "resposta": r["resposta"],
            "fontes": " | ".join(r["fontes"]), "similaridade": r["similaridade"],
            "recusa_por_limiar": r["recusa_por_limiar"], "modelo": MODELO,
        })
        print(f"[{p['id']} · {modo}] {r['resposta'][:120]}")

os.makedirs("resultados", exist_ok=True)
saida = f"resultados/rag_{MODELO}.csv"
pd.DataFrame(linhas).to_csv(saida, index=False)
print("\nSalvo em", saida, "(baixe o arquivo: a pasta some quando a sessão fecha)")